# Regularization and feature-block weighting with frozen DBSCAN labels

The first feature round held C=1000 and new feature-block weight=1 fixed. Test whether the same disease and criteria representations help when those classifier settings change. Reuse all previously fitted training-only vocabularies, IDF, embedding scaler, exact matrices, targets, and weights. No new preprocessing fit, clustering, pseudo-labeling, or feature combinations.

Grid declared before evaluation: **C=[1,10,100,1000]**, **added-block multiplier=[0.25,1,4]**, on disease minimum counts 2/3/5 and criteria minimum counts 3/50/100. Six representations × three weights × four C settings = 72 models, plus four embeddings-only controls = **76 configurations**. Multipliers apply only to the added columns; the embeddings stay unchanged. Scaling changes the relative regularization cost of using a feature block; it does not force its effect on predictions to grow by that multiplier.

Each comparison uses normalized fitting weight total 157 per reason, cutoff 0.5, highest-probability fallback, and the same 78 validation trials. Report both improvement over the original 0.910256 leader and over the embeddings-only model at the same C. Ties prefer the existing embeddings-only leader, then fewer features and settings closer to the originals. No Kaggle submission, commit, or tag.

In [1]:
from pathlib import Path
import ast
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
EMBEDDINGS=[f"emb_{i}" for i in range(150)]
C_GRID=[1.0,10.0,100.0,1000.0]
BLOCK_WEIGHTS=[0.25,1.0,4.0]
REPRESENTATIONS=["diseases_min2","diseases_min3","diseases_min5","criteria_min3","criteria_min50","criteria_min100"]
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
SOURCE=DATA/"experiments/classifier_features"
OUT=DATA/"experiments/classifier_feature_tuning"
for folder in ["datasets","models","validation_predictions","submissions"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv")
valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
truth=valid[REASONS].to_numpy(dtype=int)
original_y=train[REASONS].to_numpy(dtype=int)
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert len(train)==1171 and len(valid)==78 and known.sum()==157
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
fixed=pd.read_csv(SOURCE/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed.ID.tolist()==train.ID.tolist()
Y=fixed[REASONS].to_numpy(dtype=int);W=fixed[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original_y[known]);np.testing.assert_array_equal(W[known],np.ones((157,5)))
assert np.array_equal(W>0,Y!=-100)
(OUT/"datasets/fixed_targets_weights.csv").write_bytes((SOURCE/"datasets/fixed_targets_weights.csv").read_bytes())
source_settings=json.loads((SOURCE/"settings.json").read_text())
reference_sample=float(source_settings["reference_sample_f1"])
reference_macro=float(source_settings["reference_macro_f1"])
source_names=["embeddings_only"]+REPRESENTATIONS
SOURCE_MODELS={name:joblib.load(SOURCE/"models"/f"{name}.joblib") for name in source_names}
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",SOURCE/"settings.json",SOURCE/"comparison.csv",SOURCE/"datasets/fixed_targets_weights.csv"]
for name in source_names:protected_paths.extend([SOURCE/"models"/f"{name}.joblib",SOURCE/"datasets"/f"{name}.npz"])
additional=DATA/"experiments/additional_methods"
for rel in ["datasets/dbscan_m5_q25.csv","labeling_models/dbscan_m5_q25.joblib","classifier_models/dbscan_m5_q25__logistic_C1000.joblib"]:
    protected_paths.append(additional/rel)
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("Frozen targets, weights, embeddings, and vocabularies; 76 classifier settings declared.")


Frozen targets, weights, embeddings, and vocabularies; 76 classifier settings declared.


In [2]:
def disease_lists(frame):
    def parse(value):
        if pd.isna(value) or not str(value).strip():return []
        values=ast.literal_eval(value)
        assert isinstance(values,list)
        return sorted({str(v).strip().lower() for v in values if str(v).strip()})
    return frame.diseases_cleaned.map(parse).tolist()


def pathway_lists(frame):
    return frame.drug_pathways.fillna("").map(lambda s:sorted({v.strip() for v in str(s).split(";") if v.strip()})).tolist()


def missing_pathways(frame):
    return frame.drug_pathways.fillna("").astype(str).str.strip().eq("").to_numpy(dtype=float)[:,None]


def transform_block(block,frame):
    kind=block["kind"]
    if kind=="none":return sparse.csr_matrix((len(frame),0))
    if kind=="missing_only":return sparse.csr_matrix(missing_pathways(frame))
    if kind=="criteria":return block["vectorizer"].transform(frame.criteria.fillna("")).tocsr()
    values=disease_lists(frame) if kind=="diseases" else pathway_lists(frame)
    allowed=set(block["encoder"].classes_)
    result=block["encoder"].transform([[v for v in row if v in allowed] for row in values]).astype(float).tocsr()
    if kind=="pathways":result=sparse.hstack([result,sparse.csr_matrix(missing_pathways(frame))],format="csr")
    assert result.shape[1]==len(block["feature_names"])
    assert np.isin(result.data,[0,1]).all()
    return result


def transform_model(model,frame):
    embeddings=model["embedding_scaler"].transform(frame[model["embedding_features"]].to_numpy())
    if model["block"]["kind"]=="none":return embeddings
    return sparse.hstack([sparse.csr_matrix(embeddings),transform_block(model["block"],frame)*model.get("feature_block_weight",1.0)],format="csr")


def predict_model(model,frame):
    matrix=transform_model(model,frame)
    return np.column_stack([est.predict_proba(matrix)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    predicted=(probabilities>=0.5).astype(int)
    empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,probabilities[rows].argmax(axis=1)]=1
    return predicted,empty

reference_probability=predict_model(SOURCE_MODELS["embeddings_only"],valid)
reference_prediction,_=label_predictions(reference_probability)
np.testing.assert_allclose(f1_score(truth,reference_prediction,average="samples",zero_division=0),reference_sample,atol=1e-14)
ROWS=[];MATRIX_RECORDS={};BASELINE_SCORES={}


def prepare_matrix(representation,block_weight):
    source_matrix=sparse.load_npz(SOURCE/"datasets"/f"{representation}.npz")
    source_model=SOURCE_MODELS[representation]
    np.testing.assert_allclose(source_matrix[:,:150].toarray(),source_model["embedding_scaler"].transform(train[EMBEDDINGS].to_numpy()),rtol=1e-12,atol=1e-12)
    matrix=source_matrix.copy()
    is_block=matrix.indices>=150
    matrix.data[is_block]*=block_weight
    matrix_name=f"{representation}_weight{block_weight:g}".replace(".","p")
    sparse.save_npz(OUT/"datasets"/f"{matrix_name}.npz",matrix)
    (OUT/"datasets"/f"{matrix_name}_columns.json").write_bytes((SOURCE/"datasets"/f"{representation}_columns.json").read_bytes())
    MATRIX_RECORDS[matrix_name]={"representation":representation,"block_weight":block_weight,"shape":list(matrix.shape),
                               "source_file":str((SOURCE/"datasets"/f"{representation}.npz").relative_to(DATA)),
                               "file_sha256":hashlib.sha256((OUT/"datasets"/f"{matrix_name}.npz").read_bytes()).hexdigest()}
    return matrix,matrix_name


def fit_and_evaluate(representation,block_weight,C,matrix,matrix_name):
    name=f"{representation}_weight{block_weight:g}_C{C:g}".replace(".","p")
    template=SOURCE_MODELS[representation]
    model={**template,"C":C,"feature_block_weight":block_weight,"estimators":[],
           "training_matrix_file":f"datasets/{matrix_name}.npz","source_targets_file":"datasets/fixed_targets_weights.csv",
           "source_feature_model_file":str((SOURCE/"models"/f"{representation}.joblib").relative_to(DATA))}
    # Preserve dense input for the embeddings-only control, as in the reference experiment.
    fit_matrix=matrix.toarray() if representation=="embeddings_only" else matrix
    with threadpool_limits(limits=1):
        for j in range(5):
            estimator=clone(template["estimators"][j]).set_params(C=C)
            included=W[:,j]>0
            normalized=W[included,j]*157/W[included,j].sum()
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                estimator.fit(fit_matrix[included],Y[included,j],sample_weight=normalized)
            model["estimators"].append(estimator)
    probabilities=predict_model(model,valid)
    if C==1000 and block_weight==1:
        np.testing.assert_allclose(probabilities,predict_model(template,valid),rtol=1e-8,atol=1e-10)
    predicted,empty=label_predictions(probabilities)
    sample=f1_score(truth,predicted,average="samples",zero_division=0)
    macro=f1_score(truth,predicted,average="macro",zero_division=0)
    if representation=="embeddings_only":BASELINE_SCORES[C]=sample
    known_predicted,_=label_predictions(predict_model(model,train.loc[known]))
    row={"experiment":name,"representation":representation,"feature_group":template["block"]["kind"],
         "additional_features":len(template["block"]["feature_names"]),"total_features":matrix.shape[1],
         "C":C,"block_weight":block_weight,"matrix_file":f"datasets/{matrix_name}.npz",
         "sample_f1":sample,"macro_f1":macro,"delta_vs_current_leader":sample-reference_sample,
         "delta_vs_same_C_embeddings":sample-BASELINE_SCORES[C],
         "known_training_sample_f1":f1_score(original_y[known],known_predicted,average="samples",zero_division=0),
         "fallback_trials":int(empty.sum()),"changed_trials_vs_current":int((predicted!=reference_prediction).any(axis=1).sum()),
         **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0))),
         "is_current":bool(representation=="embeddings_only" and C==1000),
         "distance_from_current_settings":abs(float(np.log10(C/1000)))+abs(float(np.log2(block_weight))),"kaggle_score":np.nan}
    ROWS.append(row)
    frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probabilities[:,j];frame[f"predicted_{r}"]=predicted[:,j]
    frame["used_fallback"]=empty
    frame["trial_f1"]=2*(truth*predicted).sum(axis=1)/(truth.sum(axis=1)+predicted.sum(axis=1))
    frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    joblib.dump(model,OUT/"models"/f"{name}.joblib",compress=3)
    pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
    return row


def run_representation(representation):
    weights=[1.0] if representation=="embeddings_only" else BLOCK_WEIGHTS
    for weight in weights:
        matrix,matrix_name=prepare_matrix(representation,weight)
        for C in C_GRID:fit_and_evaluate(representation,weight,C,matrix,matrix_name)
    subset=pd.DataFrame(ROWS).query("representation==@representation")
    best_row=subset.sort_values("sample_f1",ascending=False).iloc[0]
    print(f"{representation}: {len(subset)} configurations, best sample F1={best_row.sample_f1:.6f}, "
          f"macro F1={best_row.macro_f1:.6f}, C={best_row.C:g}, weight={best_row.block_weight:g}, "
          f"change vs current={best_row.delta_vs_current_leader:+.6f}",flush=True)

run_representation("embeddings_only")


embeddings_only: 4 configurations, best sample F1=0.910256, macro F1=0.811421, C=1000, weight=1, change vs current=+0.000000


In [3]:
for representation in ["diseases_min2","diseases_min3","diseases_min5"]:run_representation(representation)


diseases_min2: 12 configurations, best sample F1=0.910256, macro F1=0.811421, C=1000, weight=1, change vs current=+0.000000


diseases_min3: 12 configurations, best sample F1=0.910256, macro F1=0.811421, C=1000, weight=1, change vs current=+0.000000


diseases_min5: 12 configurations, best sample F1=0.910256, macro F1=0.811421, C=1000, weight=1, change vs current=+0.000000


In [4]:
for representation in ["criteria_min3","criteria_min50","criteria_min100"]:run_representation(representation)


criteria_min3: 12 configurations, best sample F1=0.910256, macro F1=0.811421, C=1000, weight=1, change vs current=+0.000000


criteria_min50: 12 configurations, best sample F1=0.897436, macro F1=0.803884, C=1000, weight=0.25, change vs current=-0.012821


criteria_min100: 12 configurations, best sample F1=0.897436, macro F1=0.803884, C=1000, weight=0.25, change vs current=-0.012821


In [5]:
comparison=pd.DataFrame(ROWS);assert len(comparison)==76
ranked=comparison.sort_values(["sample_f1","is_current","additional_features","distance_from_current_settings","experiment"],
                             ascending=[False,False,True,True,True],kind="stable")
best=ranked.iloc[0]
best_by_representation=ranked.groupby("representation",sort=False).head(1)
best_by_group=ranked.groupby("feature_group",sort=False).head(1)
best_by_representation.to_csv(OUT/"best_by_representation.csv",index=False)
best_by_group.to_csv(OUT/"best_by_group.csv",index=False)
comparison.query("representation=='embeddings_only'").to_csv(OUT/"embedding_controls.csv",index=False)
print("Selected:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1)
display(best_by_representation[["representation","C","block_weight","additional_features","sample_f1","macro_f1","delta_vs_current_leader","delta_vs_same_C_embeddings"]].round(6))
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid)
selected_prediction,_=label_predictions(selected_probability)
# Reload and independently rescore every saved label set.
for row in comparison.itertuples(index=False):
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    predictions=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    probs=frame[[f"probability_{r}" for r in REASONS]].to_numpy()
    np.testing.assert_array_equal(label_predictions(probs)[0],predictions)
    np.testing.assert_allclose(f1_score(truth,predictions,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,predictions,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
    if row.block_weight==1 and row.C==1000:
        np.testing.assert_allclose(probs,predict_model(SOURCE_MODELS[row.representation],valid),rtol=1e-8,atol=1e-10)
# Winner preprocessing must recreate its exact saved matrix.
persisted=sparse.load_npz(OUT/best.matrix_file)
expected=sparse.csr_matrix(transform_model(selected,train))
difference=persisted-expected
assert not difference.nnz or abs(difference.data).max()<1e-12
# Independently refit winner from persisted matrix and targets/weights.
fixed_replay=pd.read_csv(OUT/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed_replay.ID.tolist()==train.ID.tolist()
np.testing.assert_array_equal(fixed_replay[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(fixed_replay[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
fit_matrix=persisted.toarray() if selected["block"]["kind"]=="none" else persisted
replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=fixed_replay[r].to_numpy(dtype=int);w=fixed_replay[f"weight_{r}"].to_numpy();included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(fit_matrix[included],y[included],sample_weight=w[included]*157/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid))[0],selected_prediction)
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest
print("All 76 saved label sets rescored; original settings reproduced; winner preprocessing/refit and source checks passed.")


Selected: embeddings_only_weight1_C1000 sample F1: 0.9102564102564102 macro F1: 0.811421460284173


,representation,C,block_weight,additional_features,sample_f1,macro_f1,delta_vs_current_leader,delta_vs_same_C_embeddings
3,embeddings_only,1000.0,1.00,0,0.910256,0.811421,0.000000,0.000000
35,diseases_min5,1000.0,1.00,71,0.910256,0.811421,0.000000,0.000000
23,diseases_min3,1000.0,1.00,145,0.910256,0.811421,0.000000,0.000000
11,diseases_min2,1000.0,1.00,249,0.910256,0.811421,0.000000,0.000000
47,criteria_min3,1000.0,1.00,2000,0.910256,0.811421,0.000000,0.000000
67,criteria_min100,1000.0,0.25,776,0.897436,0.803884,-0.012821,-0.012821
59,criteria_min50,1000.0,1.00,1533,0.897436,0.801106,-0.012821,-0.012821


All 76 saved label sets rescored; original settings reproduced; winner preprocessing/refit and source checks passed.


In [6]:
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_model(selected,test))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"C_grid":C_GRID,"block_weight_grid":BLOCK_WEIGHTS,"representations":REPRESENTATIONS,
          "configurations":76,"reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "source_experiment":"classifier_features","generated_labels_changed":False,"generated_weights_changed":False,
          "preprocessing_refitted":False,"features_added_to":"classifier only","new_feature_combinations":False,
          "embedding_scaler_fit_rows":157,"original_vocabulary_idf_fit_rows":1171,"validation_rows":78,"validation_used_for_fit":False,
          "fitting_weight_total_per_reason":157,"cutoff":0.5,"fallback":True,
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "candidate":candidate,"protected_input_sha256":protected_hashes,"matrices":MATRIX_RECORDS,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*')) if p.is_file()},
          "selection":"sample F1; ties prefer current leader, fewer features, then original settings",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"scipy":scipy.__version__,"numpy":np.__version__,"pandas":pd.__version__},
          "new_kaggle_scores":None,"reported_best_kaggle":0.87393}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
def table(frame):
    lines=["| Representation | C | Block weight | Added features | Sample F1 | Macro F1 | Change vs leader | Change vs same-C embeddings |",
           "|---|---:|---:|---:|---:|---:|---:|---:|"]
    for row in frame.itertuples(index=False):
        lines.append(f"| {row.representation} | {row.C:g} | {row.block_weight:g} | {row.additional_features} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_current_leader:+.6f} | {row.delta_vs_same_C_embeddings:+.6f} |")
    return "\n".join(lines)
lines=["# Regularization and added-feature weighting","",
       "Grid: C=1,10,100,1000; added-feature multipliers=0.25,1,4. Six existing representations (three diseases, three criteria) "
       "each have 12 settings, plus four embeddings-only controls: 76 configurations. No pathway or feature-combination tests in this round.","",
       "All embeddings, vocabularies, IDF, generated targets and weights are frozen from the previous experiment. "
       "Only classifier C and the added-column multiplier vary. Smaller C means stronger regularization. "
       "The multiplier changes relative regularization cost, not a prescribed multiplier of the feature's prediction effect. "
       "Cutoff 0.5, highest-probability fallback, and fitting-weight total 157 per reason stay fixed.","",
       f"Reference sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       f"Selected **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Best by representation","",table(best_by_representation),"",
       "## Matched embeddings-only controls","",table(comparison.query("representation=='embeddings_only'")),"",
       "## All configurations","",table(ranked),"",
       "## Verification and interpretation","",
       "All 76 saved prediction sets were reloaded and rescored. Every C=1000, weight=1 setting reproduces its previous "
       "probabilities. The winner's preprocessing recreates its exact saved feature matrix, and independent refitting "
       "from that matrix and the fixed targets/weights gives matching validation probabilities. Protected earlier artifacts retain their hashes.","",
       "For every added-feature setting, delta_vs_same_C_embeddings distinguishes an improvement due to added information "
       "from one due to changing regularization alone. Selection uses the same repeatedly consulted 78 validation trials; "
       "it does not establish independent test improvement. Models remain alternatives using one new feature group at a time.","",
       "Exact weighted sparse matrices and ordered feature names are saved once per representation/weight in datasets/, "
       "with the shared trial IDs, targets, and raw weights. Each model references its matrix and persists the original "
       "native sklearn preprocessing plus the new block weight. comparison.csv, best_by_representation.csv, "
       "best_by_group.csv, embedding_controls.csv, and settings.json preserve results and checksums.","",
       "No Kaggle submission, commit, or tag was made.",""]
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No configuration improves the primary validation metric; retain the current embeddings-only leader.")
summary="\n".join(lines)+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print("Selected:",best.experiment)
print(table(best_by_representation))
print("Candidate:",candidate)


Selected: embeddings_only_weight1_C1000
| Representation | C | Block weight | Added features | Sample F1 | Macro F1 | Change vs leader | Change vs same-C embeddings |
|---|---:|---:|---:|---:|---:|---:|---:|
| embeddings_only | 1000 | 1 | 0 | 0.910256 | 0.811421 | +0.000000 | +0.000000 |
| diseases_min5 | 1000 | 1 | 71 | 0.910256 | 0.811421 | +0.000000 | +0.000000 |
| diseases_min3 | 1000 | 1 | 145 | 0.910256 | 0.811421 | +0.000000 | +0.000000 |
| diseases_min2 | 1000 | 1 | 249 | 0.910256 | 0.811421 | +0.000000 | +0.000000 |
| criteria_min3 | 1000 | 1 | 2000 | 0.910256 | 0.811421 | +0.000000 | +0.000000 |
| criteria_min100 | 1000 | 0.25 | 776 | 0.897436 | 0.803884 | -0.012821 | -0.012821 |
| criteria_min50 | 1000 | 1 | 1533 | 0.897436 | 0.801106 | -0.012821 | -0.012821 |
Candidate: None


## Matched-control interpretation

| C | Embeddings only | Best disease addition | Best criteria addition |
|---:|---:|---:|---:|
| 1 | 0.858974 | 0.858974 | 0.867521 |
| 10 | 0.867521 | 0.867521 | 0.867521 |
| 100 | 0.888889 | 0.893162 | 0.893162 |
| 1000 | 0.910256 | 0.910256 | 0.910256 |

Nine added-feature configurations improve the embeddings-only control at the same C. The largest matched-control gain is criteria_min100 at C=1 and block weight=4: sample F1 0.867521 versus 0.858974 (+0.008547). At C=100, some disease and criteria additions improve 0.888889 to 0.893162. All remain below the existing C=1000 embeddings-only leader (0.910256). Thus these results do show modest gains over some more strongly regularized controls, but no improvement over the best current model.
